In [3]:
import pandas as pd
import zipfile
import os
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np
import ast
import time
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LeakyReLU, BatchNormalization, Dropout, Input
from tensorflow.keras.optimizers import Adam
import seaborn as sns

zip_path = '/content/archive (2).zip'
extract_path = '/content/extracted_data/'
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

csv_files = [f for f in os.listdir(extract_path) if f.endswith('.csv')]
data_file = os.path.join(extract_path, csv_files[0])
df = pd.read_csv(data_file)

df['Timestamp'] = pd.to_datetime(df['Timestamp']).astype(np.int64) // 10**9

categorical_features = ['Cognitive_State', 'Emotional_State', 'Gender', 'Session_Type', 'Environmental_Context']
numerical_features = ['GSR_Values', 'Age', 'Duration (minutes)', 'Target', 'Timestamp']

df['EEG_Frequency_Bands'] = df['EEG_Frequency_Bands'].apply(ast.literal_eval)
df['Preprocessed_Features'] = df['Preprocessed_Features'].apply(ast.literal_eval)

max_eeg_len = df['EEG_Frequency_Bands'].apply(len).max()
max_preprocessed_len = df['Preprocessed_Features'].apply(len).max()

eeg_columns = [f'EEG_{i}' for i in range(max_eeg_len)]
preprocessed_columns = [f'Preprocessed_{i}' for i in range(max_preprocessed_len)]

eeg_df = pd.DataFrame(df['EEG_Frequency_Bands'].tolist(), index=df.index, columns=eeg_columns)
preprocessed_df = pd.DataFrame(df['Preprocessed_Features'].tolist(), index=df.index, columns=preprocessed_columns)

df = df.drop(columns=['EEG_Frequency_Bands', 'Preprocessed_Features'])
df = pd.concat([df, eeg_df, preprocessed_df], axis=1)

numerical_features.extend(eeg_columns)
numerical_features.extend(preprocessed_columns)

target = df['Target']
features_df = df.drop(columns=['Target', 'Student_ID'])

categorical_features = [col for col in categorical_features if col in features_df.columns]
numerical_features = [col for col in numerical_features if col in features_df.columns]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
preprocessed_data = pipeline.fit_transform(features_df)

latent_dim = 100
data_shape = preprocessed_data.shape[1]

generator = Sequential(name='Generator')
generator.add(Dense(256, input_dim=latent_dim))
generator.add(LeakyReLU(alpha=0.2))
generator.add(BatchNormalization(momentum=0.8))
generator.add(Dense(512))
generator.add(LeakyReLU(alpha=0.2))
generator.add(BatchNormalization(momentum=0.8))
generator.add(Dense(1024))
generator.add(LeakyReLU(alpha=0.2))
generator.add(BatchNormalization(momentum=0.8))
generator.add(Dense(data_shape, activation='linear'))

discriminator = Sequential(name='Discriminator')
discriminator.add(Dense(512, input_shape=(data_shape,)))
discriminator.add(LeakyReLU(alpha=0.2))
discriminator.add(Dropout(0.2))
discriminator.add(Dense(256))
discriminator.add(LeakyReLU(alpha=0.2))
discriminator.add(Dropout(0.2))
discriminator.add(Dense(128))
discriminator.add(LeakyReLU(alpha=0.2))
discriminator.add(Dropout(0.2))
discriminator.add(Dense(1, activation='sigmoid'))

discriminator.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5), metrics=['accuracy'])

discriminator.trainable = False
gan_input = Input(shape=(latent_dim,))
fake_data = generator(gan_input)
gan_output = discriminator(fake_data)
gan = Model(gan_input, gan_output, name='GAN')
gan.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))

epochs = 10000
batch_size = 32
save_interval = 1000

X_train = preprocessed_data
start_time = time.time()

for epoch in range(epochs):
    discriminator.trainable = True
    idx = np.random.randint(0, X_train.shape[0], batch_size)
    real_data = X_train[idx]
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    fake_data = generator.predict(noise, verbose=0)
    real_labels = np.ones((batch_size, 1))
    fake_labels = np.zeros((batch_size, 1))
    d_loss_real = discriminator.train_on_batch(real_data, real_labels)
    d_loss_fake = discriminator.train_on_batch(fake_data, fake_labels)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
    discriminator.trainable = False
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    valid_labels = np.ones((batch_size, 1))
    g_loss = gan.train_on_batch(noise, valid_labels)

    if epoch % save_interval == 0:
        elapsed_time = time.time() - start_time
        print(f"Epoch {epoch}/{epochs} | D loss: {d_loss[0]:.4f}, acc.: {100*d_loss[1]:.2f}% | G loss: {g_loss:.4f} | Time: {elapsed_time:.2f}s")
        sample_noise = np.random.normal(0, 1, (1, latent_dim))
        generated_sample = generator.predict(sample_noise, verbose=0)
        print("Generated sample (first row):", generated_sample[0])

print("\nTraining finished.")

num_synthetic_samples = 10000
noise = np.random.normal(0, 1, (num_synthetic_samples, latent_dim))
synthetic_data = generator.predict(noise, verbose=0)
synthetic_data_original_scale = np.hstack((synthetic_data[:, :len(numerical_features)],
                                           synthetic_data[:, len(numerical_features):]))

num_transformer = pipeline.named_steps['preprocessor'].transformers_[0][1]
cat_transformer = pipeline.named_steps['preprocessor'].transformers_[1][1]

synthetic_numerical_scaled = synthetic_data[:, :len(numerical_features)]
synthetic_categorical_onehot = synthetic_data[:, len(numerical_features):]
synthetic_numerical_original_scale = num_transformer.inverse_transform(synthetic_numerical_scaled)
synthetic_categorical_original = cat_transformer.inverse_transform(synthetic_categorical_onehot)
synthetic_data_original_scale = np.hstack((synthetic_numerical_original_scale, synthetic_categorical_original))

synthetic_df = pd.DataFrame(synthetic_data_original_scale, columns=numerical_features + categorical_features)

integer_cols = ['Age', 'Duration (minutes)', 'Timestamp']
for col in integer_cols:
    if col in synthetic_df.columns:
        synthetic_df.loc[:, col] = pd.to_numeric(synthetic_df[col], errors='coerce')
        synthetic_df.loc[:, col] = synthetic_df[col].dropna().round().astype('Int64')

numerical_cols_to_compare = ['GSR_Values', 'Age', 'Duration (minutes)', 'EEG_0', 'Preprocessed_0']
categorical_cols_to_compare = ['Cognitive_State', 'Emotional_State', 'Gender']

print("\n--- Evaluation of Generated Data ---")

for col in numerical_cols_to_compare:
    if col in df.columns and col in synthetic_df.columns:
        print(f"\nColumn: {col}")
        print("Real Data Stats:")
        display(df[col].describe())
        print("Synthetic Data Stats:")
        display(synthetic_df[col].replace([np.inf, -np.inf], np.nan).dropna().describe())

for col in categorical_cols_to_compare:
    if col in df.columns and col in synthetic_df.columns:
        print(f"\nColumn: {col}")
        print("Real Data Counts:")
        display(df[col].value_counts())
        print("Synthetic Data Counts:")
        display(synthetic_df[col].replace([np.inf, -np.inf], np.nan).dropna().value_counts())

plt.figure(figsize=(25, 5))
for i, col in enumerate(numerical_cols_to_compare):
    if col in df.columns and col in synthetic_df.columns:
        plt.subplot(1, len(numerical_cols_to_compare), i + 1)
        sns.histplot(df[col].dropna(), kde=True, color='blue', label='Real', stat='density', common_norm=False)
        sns.histplot(synthetic_df[col].replace([np.inf, -np.inf], np.nan).dropna(), kde=True, color='orange', label='Synthetic', stat='density', common_norm=False)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 5))
for i, col in enumerate(categorical_cols_to_compare):
    if col in df.columns and col in synthetic_df.columns:
        plt.subplot(1, len(categorical_cols_to_compare), i + 1)
        real_counts = df[col].value_counts().reset_index()
        real_counts.columns = ['Category', 'Count']
        real_counts['Type'] = 'Real'
        synthetic_counts = synthetic_df[col].replace([np.inf, -np.inf], np.nan).dropna().value_counts().reset_index()
        synthetic_counts.columns = ['Category', 'Count']
        synthetic_counts['Type'] = 'Synthetic'
        combined_counts = pd.concat([real_counts, synthetic_counts], ignore_index=True)
        sns.barplot(x='Category', y='Count', hue='Type', data=combined_counts)
        plt.title(f'Frequency of {col}')
        plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

synthetic_data_filename = "synthetic_mental_health_data.csv"
synthetic_df.to_csv(synthetic_data_filename, index=False)
print(f"Synthetic data saved to {synthetic_data_filename}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 0/10000 | D loss: 0.6771, acc.: 55.47% | G loss: 0.5837 | Time: 5.19s
Generated sample (first row): [-0.342031    0.21678907 -0.28156283 -0.38281313  0.00790274  0.22826655
  0.13696527  0.25708678 -0.04949711  0.02019641 -0.3384807  -0.3854663
 -0.16152532 -0.20283774  0.03191661  0.0146802   0.24026147  0.10229251
 -0.21484247  0.25750777 -0.03855944  0.5174601  -0.46456417  0.2900907 ]
Epoch 1000/10000 | D loss: 0.4123, acc.: 82.89% | G loss: 2.0564 | Time: 115.54s
Generated sample (first row): [-0.2865107   0.79837847 -3.067809   -0.6894055  -0.15408754  0.80487823
  0.3594743  -2.5894136  -1.1119226   1.4114884   0.33855066  0.53701645
 -0.0170645   0.15931761  0.8682791   0.07477425 -0.15553215  0.87362045
  0.03166864  0.6676558  -0.4197457   0.6507335   0.6674615   0.3884134 ]
Epoch 2000/10000 | D loss: 0.3487, acc.: 86.03% | G loss: 2.7273 | Time: 225.43s
Generated sample (first row): [ 0.04689565 -1.202646    1.9225062   1.6356012   0.17840648  1.4996253
  0.88178074 -0

KeyboardInterrupt: 

In [ ]:
from google.colab import files

print("Please upload the 'archive (2).zip' file:")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


In [ ]:
import pandas as pd

synthetic_df_loaded = pd.read_csv('synthetic_mental_health_data.csv')
display(synthetic_df_loaded.head())